# Plant Disease Detection with Explainable AI (XAI)
### Interactive Exploratory Data Analysis & Grad-CAM Visualization

This notebook demonstrates:
1. Loading the stratified PlantVillage dataset splits.
2. Auditing class balance and sample distributions.
3. Running inference with fine-tuned deep learning models.
4. Generating and inspecting Grad-CAM and Grad-CAM++ saliency heatmaps.
5. Verifying explanation faithfulness via quantitative Deletion Tests.

In [ ]:
import os
import sys
from pathlib import Path
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Ensure project root is on sys.path
sys.path.append(str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

from src.config import config
from src.models import get_model, get_target_layer_for_cam
from src.gradcam import GradCAM, overlay_cam_on_image, run_deletion_test
from src.dataset import preprocess_single_image

print('PyTorch Version:', torch.__version__)
print('Device:', config.device)

## 1. Dataset Split Verification

In [ ]:
train_df = pd.read_csv(config.train_csv)
val_df = pd.read_csv(config.val_csv)
test_df = pd.read_csv(config.test_csv)

print(f"Train samples: {len(train_df):,}")
print(f"Val samples:   {len(val_df):,}")
print(f"Test samples:  {len(test_df):,}")
print(f"Total:         {len(train_df) + len(val_df) + len(test_df):,}")

## 2. Load Model & Generate Grad-CAM Saliency Maps

In [ ]:
# Load class mapping
with open(config.classes_json, 'r') as f:
    class_to_idx = json.load(f)
idx_to_class = {v: k for k, v in class_to_idx.items()}

# Load ResNet-50
model = get_model('resnet50', num_classes=len(class_to_idx), pretrained=False)
ckpt_path = config.models_dir / 'resnet50_best.pt'
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location='cpu')
    model.load_state_dict(ckpt['model_state_dict'])
    print('Loaded trained checkpoint!')
else:
    print('Checkpoint not found, loading pretrained weights...')
    model = get_model('resnet50', num_classes=len(class_to_idx), pretrained=True)

model.eval()
target_layer = get_target_layer_for_cam(model, 'resnet50')
cam_engine = GradCAM(model, target_layer)

In [ ]:
# Test on a sample leaf image
sample_path = 'app/assets/Apple___Apple_scab.jpg'
tensor_in, rgb_np = preprocess_single_image(sample_path)

heatmap_gc, pred_c, conf = cam_engine.generate_cam(tensor_in, method='gradcam')
heatmap_gcpp, _, _ = cam_engine.generate_cam(tensor_in, method='gradcam++')

overlay_gc = overlay_cam_on_image(rgb_np, heatmap_gc, alpha=0.55)
overlay_gcpp = overlay_cam_on_image(rgb_np, heatmap_gcpp, alpha=0.55)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(rgb_np)
axes[0].set_title('Original Leaf')
axes[0].axis('off')

axes[1].imshow(heatmap_gc, cmap='jet')
axes[1].set_title('Grad-CAM Heatmap')
axes[1].axis('off')

axes[2].imshow(overlay_gc)
axes[2].set_title(f'Grad-CAM Overlay ({idx_to_class[pred_c]}\nConf: {conf*100:.1f}%)')
axes[2].axis('off')

axes[3].imshow(overlay_gcpp)
axes[3].set_title('Grad-CAM++ Overlay')
axes[3].axis('off')

plt.tight_layout()
plt.show()

## 3. Quantitative Faithfulness Deletion Test

In [ ]:
del_results = run_deletion_test(model, cam_engine, tensor_in, mask_fractions=[0.1, 0.2, 0.3, 0.4, 0.5])

plt.figure(figsize=(8, 4))
pcts = [f * 100 for f in del_results['mask_fractions']]
plt.plot(pcts, del_results['salient_drops'], marker='o', color='#D32F2F', linewidth=2.5, label='Salient Masking (CAM)')
plt.plot(pcts, del_results['random_drops'], marker='s', color='#757575', linewidth=2.0, linestyle='--', label='Random Masking (Control)')
plt.xlabel('Pixels Masked (%)')
plt.ylabel('Confidence Drop (ΔP)')
plt.title('Sanity Check: Deletion Test Faithfulness')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.show()

print('Is attribution faithful?:', del_results['is_faithful'])